# OMNI-713 · SM-1 on VIRAT, on Colab's GPU

Runs the repo's own code, imported from a clone of the branch, never copied here: the measuring
(`scripts/measure_sm1.py`), fusion and the classifier (`omnisight_common/fusion.py`), both text encoders
(`services/ai_workers/app/`).

1. **Setup:** clone the branch, install the pinned versions, mount Drive
2. **Load** both text encoders, pinned, and the classifier
3. **The archive:** all 275 videos' vectors from `MyDrive/omnisight/embeddings/virat/`, refused if not whole
4. **The labels:** must give OMNI-709's chance level, before any search
5. **SM-1:** every query of `datasets/eval/sm1_queries.csv`, four modes, exact search
6. **The result** as measured
7. **Save** to `MyDrive/omnisight/results/sm1_result.json`

**Run once on a GPU:** Runtime → Change runtime type → GPU. Then Runtime → Run all. A failure is a
result: no query changes and no weight is tuned on them.


In [ ]:
# 1 · Setup
BRANCH = "slice-07-search"
REPO = "https://github.com/OmniSight-team/omni-sight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

# The versions in uv.lock; Colab keeps its own GPU build of torch and torchvision.
!pip install -q "transformers==5.20.1" "huggingface-hub==2.2.0" "safetensors==0.8.0" "open-clip-torch==3.3.0" -e /content/omni-sight/services/common

# The repo's code straight from the clone (an editable install is only seen after a
# runtime restart, so the paths are added here too).
sys.path.insert(0, "/content/omni-sight/services/common")
sys.path.insert(0, "/content/omni-sight/services/ai_workers")
sys.path.insert(0, "/content/omni-sight/scripts")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 2 · Both text encoders, pinned (D-30, D-33), and the classifier (D-38)
from pathlib import Path
from omnisight_common.fusion import CLASSIFIER_FILE, ModalityClassifier
from omnisight_common.settings import CommonSettings
from app.encoder import FrameTextEncoder, ensure_checkpoint
from app import clip_encoder
from app.clip_encoder import ClipTextEncoder
import encode_virat_archive as archive_runner
import measure_sm1 as sm1

REPO_DIR = Path("/content/omni-sight")
DRIVE = Path("/content/drive/MyDrive/omnisight")
MODELS = DRIVE / "models" / "pretrained"
VIRAT = DRIVE / "data" / "virat"
ARCHIVE = DRIVE / "embeddings" / "virat"

pins = {name: CommonSettings.model_fields[name].default for name in
        ("frame_model_id", "frame_model_revision", "clip_model_id", "clip_model_revision")}
print("pins:", pins)
siglip_dir = ensure_checkpoint(pins["frame_model_id"], pins["frame_model_revision"], MODELS / "siglip2-so400m-patch14-384")
siglip = FrameTextEncoder(siglip_dir, device="cuda")
iv2_dir = ensure_checkpoint(pins["clip_model_id"], pins["clip_model_revision"], MODELS / "internvideo2-clip-s", clip_encoder.ALLOW_PATTERNS)
iv2 = ClipTextEncoder(iv2_dir, device="cuda")
classifier = ModalityClassifier.load(pins["frame_model_id"], pins["frame_model_revision"])
print("both text encoders and the classifier loaded")


In [ ]:
# 3 · The archive: every video's vectors; refused if not whole or made with other checkpoints
queries = sm1.read_queries(REPO_DIR / "datasets" / "eval" / "sm1_queries.csv")
KEYS = archive_runner.evaluation_keys(REPO_DIR)
assert len(KEYS) == 275 and len(queries) == 50
loaded = sm1.load_archive(KEYS, ARCHIVE, VIRAT, {q["answer_rule"] for q in queries}, pins=pins)
print(f"{len(KEYS)} videos, {len(loaded):,} frames, frame vectors {loaded.frame_vectors.shape}, clip vectors {loaded.clip_vectors.shape}")
assert len(loaded) == 35_704, "OMNI-708 archived 35,704 admitted frames"


In [ ]:
# 4 · The labels, before any search: OMNI-709's chance level (25.1%, actions 11.7%, scenes 38.5%)
chance = {q["query_id"]: sm1.chance_at_5(loaded.correct[q["answer_rule"]]) for q in queries}
average = lambda kind: sum(chance[q["query_id"]] for q in queries if q["kind"] == kind) / 25
print(f"chance recall@5: all {sum(chance.values()) / 50:.1%}, actions {average('action'):.1%}, scenes {average('scene'):.1%}")
assert (round(sum(chance.values()) / 50, 3), round(average("action"), 3), round(average("scene"), 3)) == (0.251, 0.117, 0.385)


In [ ]:
# 5 · SM-1: every query, four modes, every vector compared
import time
start = time.perf_counter()
result = sm1.measure(queries, loaded, siglip, iv2, classifier)
print(f"{result['n_queries']} queries over {result['frames']:,} frames of {result['videos']} videos in {time.perf_counter() - start:.1f} s")


In [ ]:
# 6 · The result, as measured
print(f"{'':8s}{'frame':>8s}{'clip':>8s}{'fused':>8s}{'equal':>8s}{'chance':>8s}")
for name, row in [("all", result["overall"]), ("scene", result["by_kind"]["scene"]), ("action", result["by_kind"]["action"])]:
    print(f"{name:8s}{row['frame']:8.1%}{row['clip']:8.1%}{row['fused']:8.1%}{row['fused_equal']:8.1%}{row['chance']:8.1%}")
print(f"action lift, fused over frame: {result['action_lift']:+.1%}")
print("PASS" if result["passed"] else "FAIL", f"(fused at least {result['target']:.0%} overall, and above frame on action queries)")
print()
for row in result["queries"]:
    marks = " ".join(f"{m}:{row['modes'][m]['first_correct_rank'] or '-'}" for m in sm1.MODES)
    print(f"{row['query_id']} w_f {row['w_f']:.2f}  {marks}  {row['text']}")


In [ ]:
# 7 · Save to Drive; Claude Code copies it to services/query/app/sm1_result.json for GET /metrics/sm1
import json
result["conditions"] = {
    "commit": COMMIT, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "pins": pins,
    "queries_sha256": sm1.file_sha256(REPO_DIR / "datasets" / "eval" / "sm1_queries.csv"),
    "classifier_sha256": sm1.file_sha256(CLASSIFIER_FILE),
    "k": sm1.K, "search": "exact, every vector compared",
}
out = DRIVE / "results" / "sm1_result.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(result, indent=1))
print(out, result["conditions"])
